In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation import get_forecast_folds, apply_scoring_mask, calculate_rel_mae


# Example fold generation for validation origins
validation_origins = ["2016-07-31", "2017-07-31", "2018-07-31", "2019-07-31", "2022-07-31", "2023-07-31"]
folds = get_forecast_folds(validation_origins)

for fold in folds:
    print(f"Origin: {fold['origin'].strftime('%Y-%m')} | Forecasting: {fold['forecast_months'][0].strftime('%Y-%m')} to {fold['forecast_months'][-1].strftime('%Y-%m')}")

Origin: 2016-07 | Forecasting: 2016-09 to 2017-08
Origin: 2017-07 | Forecasting: 2017-09 to 2018-08
Origin: 2018-07 | Forecasting: 2018-09 to 2019-08
Origin: 2019-07 | Forecasting: 2019-09 to 2020-08
Origin: 2022-07 | Forecasting: 2022-09 to 2023-08
Origin: 2023-07 | Forecasting: 2023-09 to 2024-08


In [4]:
import pandas as pd
import numpy as np
from src.evaluation import get_forecast_folds, apply_scoring_mask, calculate_mae, calculate_rmse, calculate_bias, calculate_rel_mae


# Load the master monthly macroeconomic and apparel sales dataset
# (Adjust the path relative to your notebook location if needed, e.g., "../reports/...")
from src.analysis.eda_helper import load_apparel_data

df = load_apparel_data()


# Ensure the index is set to monthly start frequency (MS)
df = df.asfreq("MS")

# Verify the DataFrame is loaded correctly
print("Dataset shape:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())


# 1. Setup validation origins
validation_origins = [
    "2016-07-31", "2017-07-31", "2018-07-31", 
    "2019-07-31", "2022-07-31", "2023-07-31"
]
folds = get_forecast_folds(validation_origins)

results = []

for fold in folds:
    origin = fold["origin"]
    forecast_months = fold["forecast_months"]
    
    # Extract actuals for target window
    y_true = df.loc[forecast_months, "apparel_sales_nsa"]
    
    # Baseline 1: Seasonal Naïve (exact values from 12 months prior)
    y_pred_sn = df.loc[forecast_months - pd.DateOffset(years=1), "apparel_sales_nsa"].values
    y_pred_sn = pd.Series(y_pred_sn, index=y_true.index)
    
    # Baseline 2: Seasonal Naïve + Drift (adding historical average YoY growth rate)
    # Drift calculated strictly on training data up to origin
    train_data = df.loc[df.index <= origin, "apparel_sales_nsa"]
    yoy_growth_mean = train_data.pct_change(12).mean()
    y_pred_drift = y_pred_sn * (1 + yoy_growth_mean)
    
    # Apply scoring mask
    y_true_masked, y_pred_sn_masked = apply_scoring_mask(y_true, y_pred_sn)
    _, y_pred_drift_masked = apply_scoring_mask(y_true, y_pred_drift)
    
    # Isolate December error specifically
    dec_month = pd.Timestamp(f"{origin.year + 1}-12-01") if origin.month == 7 else None # Dec falls inside horizon
    # More robust December extraction across the 12-month horizon:
    dec_dates = [m for m in y_true.index if m.month == 12]
    dec_error_sn = float(np.mean(np.abs(y_pred_sn.loc[dec_dates] - y_true.loc[dec_dates]))) if dec_dates else np.nan
    dec_error_drift = float(np.mean(np.abs(y_pred_drift.loc[dec_dates] - y_true.loc[dec_dates]))) if dec_dates else np.nan

    results.append({
        "Origin": origin.strftime("%Y-%m"),
        "SN_MAE": calculate_mae(y_true_masked, y_pred_sn_masked),
        "SN_RMSE": calculate_rmse(y_true_masked, y_pred_sn_masked),
        "SN_Bias": calculate_bias(y_true_masked, y_pred_sn_masked),
        "SN_Dec_MAE": dec_error_sn,
        "Drift_RelMAE": calculate_rel_mae(y_true_masked, y_pred_drift_masked, y_pred_sn_masked),
        "Drift_MAE": calculate_mae(y_true_masked, y_pred_drift_masked),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Dataset shape: (415, 6)
Date range: 1992-01-01 00:00:00 to 2026-07-01 00:00:00
 Origin     SN_MAE    SN_RMSE     SN_Bias  SN_Dec_MAE  Drift_RelMAE   Drift_MAE
2016-07 455.333333 570.094583 -238.166667      1203.0      1.103774  502.585153
2017-07 588.583333 717.866689 -259.083333       859.0      1.068690  629.012864
2018-07 425.750000 521.220283 -198.416667       116.0      1.343218  571.875102
2019-07 577.833333 703.425547 -440.166667      1089.0      0.851741  492.164409
2022-07 554.083333 671.284651 -170.250000       442.0      2.321844 1286.494898
2023-07 625.500000 767.372139 -535.833333      1386.0      1.384418  865.953514


### Model Evaluation Findings & Strategic Takeaways

### 📊 Summary of Baseline Performance

### 1. Systematic Under-Forecasting (Negative Bias)
* **Finding:** Every single fold exhibits a negative `SN_Bias`, ranging from **-\$170M to -\$535M**.
* **Interpretation:** Because the Seasonal Naïve baseline assumes next year's sales will exactly match this year (\(\hat{Y}_{t+12} = Y_t\)), it completely misses long-term secular sales growth and baseline inflation. Sales consistently grow year-over-year, causing the naïve baseline to chronically under-forecast.
* **💡 Model Takeaway:** Advanced models (**SARIMA / Prophet**) must incorporate an explicit trend component rather than relying purely on stationary differencing or static copying.

---

### 2. The Failure of Simple Global Drift
* **Finding:** The `Drift_RelMAE` is greater than **1.0 in 5 out of 6 folds**, and explodes to **2.32** in the July 2022 origin. The only fold where drift helped was July 2019 (\(\text{RelMAE} = 0.85\)).
* **Interpretation:** Adding a static, global historical growth rate to a seasonal baseline causes massive over-corrections when macroeconomic regimes shift. In July 2022 (coming out of the post-COVID inflationary wave), applying a linear drift rate caused errors to more than double (**\$1,286M MAE**).
* **💡 Model Takeaway:** Global linear drift is unstable. We need models that dynamically adapt their trend via localized parameter estimation or state-space updating.

---

### 3. December Holiday Volatility
* **Finding:** `SN_Dec_MAE` spikes as high as **\$1,386M** (2023 origin), dwarfs standard monthly errors, and fluctuates wildly (dropping anomalously to **\$116M** in 2018).
* **Interpretation:** The December holiday peak is highly sensitive to calendar shifts, weekend placements, and consumer spending timing. A generic seasonal baseline struggles to capture the shifting magnitude of the holiday spike.
* **💡 Model Takeaway:** We must evaluate future models heavily on December-specific error to ensure our inventory and temporary staffing plans remain secure.


### Sanity Checks & Diagnostic Evaluation

### 🔍 Validation Safeguards & Benchmarking

### 1. Behavior of the 2019 Fold & COVID Interaction
* **Forecast Horizon:** The July 2019 validation origin targets **August 2019 through July 2020**.
* **The COVID Intersection:** This target window directly intersects with the onset of the pandemic in **March–July 2020**.
* **Sanity Validation:** Because our evaluation harness enforces the scoring mask (**2020-03-01 through 2021-06-30**), the catastrophic pandemic lockdown months are automatically excluded from the MAE and RMSE calculations for the 2019 fold. The metrics for 2019 reflect normal pre-COVID operations (**Aug 2019 – Feb 2020**) plus any post-mask periods, ensuring the baseline numbers are not artificially distorted by a once-in-a-century macroeconomic shock.

---

### 2. Drift vs. Plain Seasonal Naïve
* **Empirical Finding:** Looking at the `Drift_RelMAE` column across our 6 folds:
  * **2016:** 1.10 *(Worse)*
  * **2017:** 1.07 *(Worse)*
  * **2018:** 1.34 *(Worse)*
  * **2019:** 0.85 *(Better)*
  * **2022:** 2.32 *(Significantly Worse)*
  * **2023:** 1.38 *(Worse)*
* **Interpretation:** Simple global linear drift performs worse than plain Seasonal Naïve in **5 out of 6 folds**. Applying a static historical growth rate to a seasonal series causes severe overcorrections when economic conditions shift (most notably in 2022, where drift more than doubled the error to a RelMAE of 2.32).
* **💡 Modeling Bar:** Because plain Seasonal Naïve is superior to the drift version, our benchmark success bar (**RelMAE ≤ 0.90**) remains locked against plain Seasonal Naïve.

---

### 3. Metric Noise & Fold Consistency
* **Sample Size Risk:** Evaluating across only **6 validation folds** means our performance metrics are susceptible to macro-level noise (e.g., individual inflationary waves or supply chain anomalies like the 2022 origin).
* **Consistency Rule:** An average RelMAE can be easily skewed by a single lucky fold. To claim success, our advanced models (**SARIMA / Prophet**) must demonstrate structural consistency—winning across a majority of the folds (e.g., maintaining **RelMAE ≤ 0.90 in at least 5 out of 6 folds**) rather than relying on one volatile year to pull down the aggregate average.
